# Analisis Sentimen ESG — LinkedIn Perusahaan Tambang Indonesia
---
**Notebook:** 02_sentiment_analysis  
**Deskripsi:** Lexicon-based Sentiment Analysis pada postingan LinkedIn terkait ESG perusahaan tambang.

Pipeline:
1. Load data processed CSV
2. Lexicon-based sentiment (TextBlob + InSet Indonesia)
3. Visualisasi distribusi, time series, word cloud
4. Export summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from textblob import TextBlob
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Paths
DATA_DIR = Path('../data')
RESULT_DIR = Path('../results')
FIGURE_DIR = Path('../paper/figures')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
# Load processed CSV
csv_files = list(DATA_DIR.glob('processed/*_processed.csv'))
if not csv_files:
    # Fallback: original project location
    csv_files = list(Path('../../clean').glob('*_processed.csv'))
    
if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f'Loaded: {csv_files[0].name}')
else:
    print('No processed CSV found. Using sample...')
    import json
    raw_files = list(DATA_DIR.glob('raw/*_offline_raw.json'))
    if raw_files:
        with open(raw_files[0]) as f:
            posts = json.load(f)
        rows = [{
            'company': p.get('company'),
            'post_id': p.get('post_id'),
            'text_raw': p.get('text_raw', ''),
            'text_clean': re.sub(r'https?://\\S+', ' ', p.get('text_raw', '')).strip().lower(),
            'posted_at_utc': p.get('posted_at_utc'),
            'reactions_count': p.get('reactions_count'),
            'comments_count': p.get('comments_count'),
            'reshares_count': p.get('reshares_count'),
        } for p in posts]
        df = pd.DataFrame(rows)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
if 'company' in df.columns:
    print(f'Companies: {df["company"].unique()}')

In [ ]:
df.head(2)

## 2. Lexicon-based Sentiment Analysis

In [ ]:
# ── InSet Lexicon for Bahasa Indonesia ──
# Positive words with scores
IN_SET_POSITIF = {
    'baik': 3, 'bagus': 3, 'hebat': 4, 'luar biasa': 5, 'inovatif': 3,
    'berkelanjutan': 3, 'ramah lingkungan': 4, 'hijau': 2, 'bersih': 3,
    'maju': 3, 'sukses': 4, 'berhasil': 3, 'prestasi': 4, 'penghargaan': 3,
    'bermanfaat': 3, 'membantu': 2, 'peduli': 3, 'memberdayakan': 3,
    'berdaya': 2, 'mandiri': 3, 'sejahtera': 4, 'mensejahterakan': 4,
    'transparan': 3, 'akuntabel': 3, 'tepercaya': 3, 'komitmen': 2,
    'dedikasi': 3, 'kolaborasi': 2, 'kemitraan': 2, 'sinergi': 3,
    'apresiasi': 3, 'terima kasih': 2, 'bangga': 4, 'bersyukur': 3,
    'cerdas': 3, 'efisien': 3, 'efektif': 3, 'modern': 2, 'canggih': 3,
    'pelopor': 4, 'terdepan': 3, 'unggul': 4, 'kualitas': 2,
}

# Negative words with scores
IN_SET_NEGATIF = {
    'buruk': -3, 'jelek': -3, 'gagal': -4, 'kegagalan': -4,
    'korupsi': -5, 'suap': -5, 'pencemaran': -4, 'polusi': -3,
    'limbah berbahaya': -4, 'deforestasi': -3, 'kerusakan': -4,
    'konflik': -3, 'sengketa': -3, 'demo': -3, 'protes': -3,
    'pelanggaran': -4, 'melanggar': -4, 'lalai': -3, 'kelalaian': -3,
    'kecelakaan': -4, 'celaka': -4, 'korban': -3, 'meninggal': -4,
    'PHK': -4, 'pemecatan': -3, 'pengangguran': -2, 'kemiskinan': -3,
    'krisis': -3, 'ancaman': -3, 'mengancam': -3, 'rusak': -3,
    'merusak': -4, 'mencemari': -4, 'eksploitasi': -4, 'mengeksploitasi': -4,
}


def textblob_score(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return TextBlob(text).sentiment.polarity


def inset_score(text):
    """Score using InSet lexicon for Bahasa Indonesia."""
    if not isinstance(text, str) or not text.strip():
        return 0.0
    t = text.lower()
    score = 0.0
    for word, val in IN_SET_POSITIF.items():
        if word in t:
            score += val
    for word, val in IN_SET_NEGATIF.items():
        if word in t:
            score += val
    return score


def classify_sentiment(score):
    if score > 0.1:
        return 'Positif'
    elif score < -0.1:
        return 'Negatif'
    else:
        return 'Netral'


# Apply TextBlob
df['polarity_tb'] = df['text_raw'].apply(textblob_score)

# Apply InSet (normalized to -1..1 scale)
df['polarity_inset'] = df['text_raw'].apply(inset_score) / 10.0
df['polarity_inset'] = df['polarity_inset'].clip(-1, 1)

# Combined score
df['polarity'] = (df['polarity_tb'] + df['polarity_inset']) / 2
df['sentiment'] = df['polarity'].apply(classify_sentiment)

print(df['sentiment'].value_counts())
print(f"\nMean polarity (combined): {df['polarity'].mean():.3f}")

In [ ]:
# Examples
for sent in ['Positif', 'Netral', 'Negatif']:
    sample = df[df['sentiment'] == sent]['text_raw'].iloc[0] if len(df[df['sentiment'] == sent]) > 0 else 'N/A'
    print(f'=== {sent} ===')
    print(sample[:200])
    print()

## 3. Visualisasi Distribusi Sentimen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie
sent_counts = df['sentiment'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#95a5a6']
axes[0].pie(sent_counts.values, labels=sent_counts.index, autopct='%1.1f%%',
            colors=colors[:len(sent_counts)], startangle=90)
axes[0].set_title('Distribusi Sentimen')

# Bar per company
if 'company_norm' in df.columns:
    company_col = 'company_norm'
elif 'company' in df.columns:
    company_col = 'company'
else:
    company_col = None

if company_col:
    sent_by_company = df.groupby([company_col, 'sentiment']).size().unstack(fill_value=0)
    sent_by_company.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
    axes[1].set_title('Sentimen per Perusahaan')
    axes[1].set_xlabel('Perusahaan')
    axes[1].set_ylabel('Jumlah Postingan')
    axes[1].legend(title='Sentimen')

plt.tight_layout()
plt.savefig(FIGURE_DIR / '01_distribusi_sentimen.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Time Series
if 'month' in df.columns and 'sentiment' in df.columns:
    try:
        ts = df.groupby(['month', 'sentiment']).size().unstack(fill_value=0)
        fig, ax = plt.subplots(figsize=(14, 5))
        ts.plot(kind='line', marker='o', ax=ax, linewidth=2)
        ax.set_title('Tren Sentimen per Bulan')
        ax.set_xlabel('Bulan')
        ax.set_ylabel('Jumlah Postingan')
        ax.legend(title='Sentimen')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / '02_tren_sentimen.png', dpi=300, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f'Time series plot skipped: {e}')
else:
    print('Column "month" or "sentiment" not found')

In [ ]:
# Word Cloud
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    for idx, sentiment in enumerate(['Positif', 'Negatif']):
        text = ' '.join(df[df['sentiment'] == sentiment]['text_clean'].dropna()) if 'text_clean' in df.columns else ''
        if not text.strip():
            text = ' '.join(df[df['sentiment'] == sentiment]['text_raw'].dropna())
        if text.strip():
            wc = WordCloud(width=800, height=400, background_color='white',
                          colormap='viridis' if sentiment == 'Positif' else 'Reds',
                          max_words=50).generate(text)
            axes[idx].imshow(wc, interpolation='bilinear')
            axes[idx].axis('off')
            axes[idx].set_title(f'Word Cloud — {sentiment}', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / '03_wordcloud.png', dpi=300, bbox_inches='tight')
    plt.show()
except ImportError:
    print('wordcloud not installed. Install: pip install wordcloud')

In [ ]:
# Engagement vs Sentimen
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['reactions_count', 'comments_count', 'reshares_count']
titles = ['Reactions vs Sentimen', 'Comments vs Sentimen', 'Reshares vs Sentimen']
for idx, (metric, title) in enumerate(zip(metrics, titles)):
    if metric in df.columns:
        df[metric] = pd.to_numeric(df[metric], errors='coerce').fillna(0)
        sns.boxplot(data=df, x='sentiment', y=metric, ax=axes[idx],
                   palette=colors, order=['Positif', 'Netral', 'Negatif'])
        axes[idx].set_title(title)
        axes[idx].set_ylabel(metric)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '04_engagement_vs_sentimen.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. ESG Categorization

In [ ]:
# ESG categorization
ESG_KEYWORDS = {
    'E': ['lingkungan', 'environment', 'carbon', 'emisi', 'emission', 'climate', 'iklim',
          'energi', 'energy', 'renewable', 'terbarukan', 'reklamasi', 'reclamation',
          'limbah', 'waste', 'polusi', 'biodiversity', 'green', 'hijau', 'net zero',
          'decarbonization', 'dekarbonisasi'],
    'S': ['sosial', 'social', 'community', 'masyarakat', 'csr', 'pekerja', 'worker',
          'employee', 'karyawan', 'k3', 'safety', 'keselamatan', 'health', 'kesehatan',
          'human rights', 'hak asasi', 'diversity', 'inklusif', 'inclusion',
          'pendidikan', 'education', 'pemberdayaan', 'empowerment'],
    'G': ['governance', 'tata kelola', 'transparansi', 'transparency', 'anti korupsi',
          'anticorruption', 'audit', 'compliance', 'kepatuhan', 'dewan direksi',
          'board', 'executive', 'whistleblower', 'etika', 'ethics', 'shareholder',
          'pemegang saham', 'stakeholder'],
}

def categorize_esg(text):
    if not isinstance(text, str):
        return []
    t = text.lower()
    found = []
    for pillar, keywords in ESG_KEYWORDS.items():
        for kw in keywords:
            if kw in t:
                found.append(pillar)
                break
    return found

df['esg_pillars'] = df['text_raw'].apply(categorize_esg)
df['esg_label'] = df['esg_pillars'].apply(lambda x: '/'.join(x) if x else 'None')

print('=== ESG Distribution ===')
print(df['esg_label'].value_counts())

fig, ax = plt.subplots(figsize=(10, 5))
if company_col:
    pivot = df.groupby([company_col, 'esg_label']).size().unstack(fill_value=0)
    pivot.plot(kind='bar', ax=ax, edgecolor='white')
    ax.set_title('ESG Distribution per Company')
    ax.set_xlabel('Company')
    ax.set_ylabel('Post Count')
    ax.legend(title='ESG Pillar')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / '05_esg_pillar_dist.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Export Summary

In [ ]:
# Summary statistics
summary = {
    'total_posts': len(df),
    'positive_pct': (df['sentiment'] == 'Positif').mean() * 100,
    'negative_pct': (df['sentiment'] == 'Negatif').mean() * 100,
    'neutral_pct': (df['sentiment'] == 'Netral').mean() * 100,
    'mean_polarity': df['polarity'].mean(),
}
summary_df = pd.DataFrame([summary])
summary_df.to_csv(RESULT_DIR / 'sentiment_summary.csv', index=False)
print(summary_df.T)
print(f'\nSaved to: {RESULT_DIR / "sentiment_summary.csv"}')

In [ ]:
# Per-company breakdown
if company_col:
    per_company = df.groupby(company_col).agg(
        total=('sentiment', 'count'),
        positif=('sentiment', lambda x: (x == 'Positif').sum()),
        negatif=('sentiment', lambda x: (x == 'Negatif').sum()),
        netral=('sentiment', lambda x: (x == 'Netral').sum()),
        mean_polarity=('polarity', 'mean'),
        avg_reactions=('reactions_count', 'mean'),
    ).reset_index()
    per_company['positif_pct'] = (per_company['positif'] / per_company['total'] * 100).round(1)
    per_company['negatif_pct'] = (per_company['negatif'] / per_company['total'] * 100).round(1)
    per_company.to_csv(RESULT_DIR / 'sentiment_per_company.csv', index=False)
    print(per_company)
    print(f'\nSaved to: {RESULT_DIR / "sentiment_per_company.csv"}')

In [ ]:
# Extreme sentiment posts for thematic analysis
top_positif = df.nlargest(50, 'polarity')
top_negatif = df.nsmallest(50, 'polarity')
extreme = pd.concat([top_positif, top_negatif]).drop_duplicates(subset=['post_id'])
cols = [c for c in ['company_norm', 'company', 'post_id', 'text_raw', 'polarity', 'sentiment'] if c in extreme.columns]
extreme[cols].to_csv(RESULT_DIR / 'extreme_sentiment_posts.csv', index=False)
print(f'Extreme posts for thematic analysis: {len(extreme)}')

## 6. Kesimpulan

Notebook ini menyediakan:
1. **Sentiment distribution** — pie, bar, time series
2. **Word cloud** — kata dominan per sentimen
3. **Engagement analysis** — hubungan engagement dengan sentimen
4. **ESG categorization** — distribusi E, S, G
5. **Exported outputs** — CSV summary + extreme posts untuk thematic analysis

Hasil ini menjadi input untuk **Fase Kualitatif** (03_thematic_analysis.ipynb).